# 🧠 The Complete Python → GenAI Curriculum
### *From Variables to Distributed Production Systems — Every Concept You Need to Master GenAI Engineering*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aashiq-parinda/genai-systems-portfolio/blob/main/notebooks/Complete_Python_GenAI_Curriculum.ipynb)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue.svg)](https://www.python.org/)
[![MAANG Interview Ready](https://img.shields.io/badge/MAANG-Interview%20Ready-orange.svg)](#)

---
> **This is the definitive Python curriculum for engineers who want to build production-grade Generative AI systems.**
> Every concept is taught with runnable code, practical GenAI context, and MAANG-level interview preparation.

---
## 🗺️ Curriculum Map

| Section | Theme | Topics |
|---|---|---|
| 🧱 **A** | **Core Python** | Variables, Types, Collections, Functions, Comprehensions, Mutability |
| 🧠 **B** | **Python Internals** | Object Identity, Memory, GIL, Reference Counting, Garbage Collection |
| ⚙️ **C** | **Advanced Python** | Decorators, Generators, Context Managers, Exceptions, OOP, Pydantic |
| ⚡ **D** | **Concurrency** | Threading, Multiprocessing, asyncio, Race Conditions, Locks, Pools |
| 🌐 **E** | **Backend Engineering** | HTTP, FastAPI, Auth, Rate Limiting, Retries, Idempotency, Caching |
| 🤖 **F** | **GenAI Engineering** | LLM APIs, Streaming, Tools, RAG, Agents, Embeddings, Context Windows |
| 🏗️ **G** | **Production Systems** | Logging, Metrics, Tracing, Observability, Resilience, Distributed Systems |

In [ ]:
# Environment Setup & Verification
import sys, os, json, time, math, copy, random, gc, threading, asyncio
import hashlib, uuid, re, csv, io, queue, functools, collections
from typing import List, Dict, Any, Optional, Union, Tuple, Generator, Callable
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
print(f"✅ Python {sys.version.split()[0]} ready on {sys.platform}")
print("✅ All standard library modules imported successfully!")

---
# 🧱 SECTION A — CORE PYTHON
*Variables, Data Types, Collections, Functions, Comprehensions, Slicing, Mutability, References*

## A1 — Variables, Data Types & None

### 📚 Textbook Definition
Python is **dynamically and strongly typed**. A variable is a label bound to an object in memory. Types include `int`, `float`, `str`, `bool`, and `NoneType`.

### 🧠 Intuition
Variables are like sticky notes labelling boxes. The box (`object`) contains the value. Two sticky notes can point to the same box.

### 💀 Common Mistake
Using `if not val:` to detect missing values when `0`, `0.0`, `""`, and `False` are all valid inputs.

In [ ]:
# ── Data Types ──────────────────────────────────────────────
model_name: str         = "gpt-4o"
temperature: float      = 0.7
max_tokens: int         = 4096
streaming: bool         = True
finish_reason: None     = None   # Will be set later

# ── Type inspection ─────────────────────────────────────────
for name, val in [("model_name", model_name), ("temperature", temperature),
                   ("max_tokens", max_tokens), ("streaming", streaming),
                   ("finish_reason", finish_reason)]:
    print(f"{name:15s} = {repr(val):15s}  type={type(val).__name__}")

# ── Safe sentinel check (the RIGHT way) ─────────────────────
def safe_default(value, default):
    """Sets default ONLY when value is truly absent (None), not when it is falsy."""
    return default if value is None else value

print("\nSafe defaults:")
print(safe_default(0.0, 0.7))    # 0.0 → keeps 0.0 (deterministic greedy decoding)
print(safe_default(None, 0.7))   # None → sets 0.7 (missing parameter)

## A2 — Strings: Operations, f-strings & Template Patterns

### 📚 Textbook Definition
Strings are **immutable** sequences of Unicode characters. Python offers f-strings, `.format()`, and various string methods for manipulation.

### 💀 Common Mistake
Using `str.replace` or concatenation inside a hot loop — creates O(N²) memory allocation. Use `str.join()` instead.

In [ ]:
# ── String construction patterns ────────────────────────────
system_prompt = "You are a {role} specialising in {domain}."
filled = system_prompt.format(role="financial analyst", domain="equity research")
print("Formatted prompt:", filled)

# f-string (fastest at runtime)
model, temp = "claude-3-5-sonnet", 0.2
config_str = f"model={model!r}, temperature={temp}"
print("Config:", config_str)

# ── Efficient multi-part assembly ────────────────────────────
turns = ["Explain RAG.", "How does chunking work?", "What is cosine similarity?"]
prompt_body = "\n".join(f"[Q{i+1}] {q}" for i, q in enumerate(turns))
print("\nPrompt body:\n" + prompt_body)

# ── Important string methods for GenAI ──────────────────────
raw = "  ```json\n{\"answer\": 42}\n```  "
stripped = raw.strip().removeprefix("```json").removesuffix("```").strip()
print("\nStripped JSON fence:", stripped)

## A3 — Lists: Indexing, Slicing & Message History Management

### 📚 Textbook Definition
Lists are **ordered, mutable** sequences. Slicing (`list[start:stop:step]`) creates a shallow copy of the slice.

### 🧠 Intuition
A chat history is a list. Context truncation is list slicing. Inserting a retrieved document is list insertion.

In [ ]:
# ── Chat history as a list ──────────────────────────────────
history = [
    {"role": "system",    "content": "Be concise."},
    {"role": "user",      "content": "What is RAG?"},
    {"role": "assistant", "content": "RAG = Retrieval + Generation."},
    {"role": "user",      "content": "Give me an example."},
    {"role": "assistant", "content": "A chatbot that searches docs before answering."},
]

# ── Slicing patterns ─────────────────────────────────────────
system_only    = history[:1]           # first element
last_2_turns   = history[-2:]          # last 2 elements
no_system      = history[1:]           # skip system prompt
reversed_turns = history[::-1]         # reversed (debug inspection)

# ── Sliding-window truncation keeping system prompt ──────────
MAX_TURNS = 3
truncated = history[:1] + history[-(MAX_TURNS*2):]
print(f"Truncated history ({len(truncated)} turns):")
for m in truncated:
    print(f"  [{m['role']:9s}]: {m['content']}")

# ── List operations ───────────────────────────────────────────
chunks = ["Chunk A", "Chunk B", "Chunk C"]
chunks.insert(1, "Injected Chunk")       # insert at index 1
removed = chunks.pop()                   # remove last
print("\nChunks after ops:", chunks)
print("Removed:", removed)

## A4 — Dictionaries: Patterns, Merging & Nested Access

### 📚 Textbook Definition
Dictionaries are **ordered** (Python 3.7+), **mutable** key-value hash maps providing O(1) average-case lookups.

### 💀 Common Mistake
Using `dict[key]` instead of `dict.get(key, default)` on optional JSON fields from an LLM response.

In [ ]:
# ── API-style response dictionary ───────────────────────────
response = {
    "id": "chatcmpl-9ABC",
    "model": "gpt-4o",
    "choices": [
        {"message": {"role": "assistant", "content": "Paris is the capital of France."},
         "finish_reason": "stop"}
    ],
    "usage": {"prompt_tokens": 18, "completion_tokens": 9, "total_tokens": 27}
}

# ── Safe deep-path access ────────────────────────────────────
content     = response.get("choices", [{}])[0].get("message", {}).get("content", "")
tool_calls  = response.get("tool_calls")  # None if absent — no KeyError
total_tok   = response["usage"]["total_tokens"]
print("Content:", content)
print("Tool calls (absent):", tool_calls)
print("Total tokens:", total_tok)

# ── Dictionary merging (override pattern) ────────────────────
base_config = {"temperature": 0.7, "max_tokens": 1024, "stream": False}
user_cfg    = {"temperature": 0.2, "stream": True}
effective   = {**base_config, **user_cfg}       # user values win
print("\nEffective config:", effective)

# ── Dict comprehension to index by field ─────────────────────
docs = [{"id": "d1", "text": "RAG grounding"}, {"id": "d2", "text": "LLM routing"}]
doc_index = {d["id"]: d for d in docs}
print("Indexed doc:", doc_index["d1"])

## A5 — Sets & Tuples

### 📚 Textbook Definition
**Sets** are unordered, mutable collections of unique hashable elements with O(1) membership tests. **Tuples** are ordered, **immutable** sequences; hashable when elements are hashable.

### 🧠 Intuition
Use a `set` to deduplicate retrieved chunks. Use a `tuple` as a cache key `(query, model_id)`.

In [ ]:
# ── Sets for deduplication ──────────────────────────────────
retrieved = ["Chunk A", "Chunk B", "Chunk A", "Chunk C", "Chunk B"]
seen = set()
unique = []
for c in retrieved:
    if c not in seen:
        seen.add(c)
        unique.append(c)
print("Unique ordered chunks:", unique)

# ── Set operations ────────────────────────────────────────────
model_a_caps = {"json_mode", "streaming", "tool_calling"}
model_b_caps = {"streaming", "vision", "tool_calling"}
both   = model_a_caps & model_b_caps   # intersection
either = model_a_caps | model_b_caps   # union
only_a = model_a_caps - model_b_caps   # difference
print("Shared capabilities:", both)
print("Only in Model A:    ", only_a)

# ── Tuples as immutable cache keys ───────────────────────────
cache = {}
def embed_with_cache(text: str, model: str) -> list:
    key = (text, model)              # hashable tuple key
    if key not in cache:
        cache[key] = [hash(text) % 100 / 100.0] * 4  # fake embedding
    return cache[key]

print("\nCached embedding:", embed_with_cache("RAG is powerful", "ada-002"))
print("Cache keys:", list(cache.keys()))

## A6 — Functions: Parameters, Return Values, Default Args, *args, **kwargs

### 📚 Textbook Definition
Functions are first-class objects. Parameters include positional, keyword, `*args` (variadic positional → `tuple`), and `**kwargs` (variadic keyword → `dict`).

### 💀 Common Mistake — Mutable Default Argument
Using `history=[]` as a default parameter — that list is shared across ALL calls.

In [ ]:
# ── Correct default with None sentinel ──────────────────────
def build_messages(prompt: str, history: Optional[list] = None,
                   system: str = "You are a helpful AI.") -> list:
    if history is None:
        history = []                     # Fresh list every call — never shared!
    return [{"role": "system", "content": system}] + history +            [{"role": "user", "content": prompt}]

print("Turn 1:", len(build_messages("Hello")), "messages")
print("Turn 2:", len(build_messages("Hi again")), "messages")  # NOT accumulated

# ── *args and **kwargs ───────────────────────────────────────
def create_payload(prompt: str, *stop_sequences: str, **model_kwargs) -> dict:
    return {
        "messages": [{"role": "user", "content": prompt}],
        "stop": list(stop_sequences),
        **model_kwargs
    }

payload = create_payload(
    "Explain LoRA",
    "<END>", "\n---",
    model="claude-3-5-sonnet",
    temperature=0.2,
    max_tokens=512
)
print("\nPayload keys:", list(payload.keys()))
print("Stop sequences:", payload["stop"])

# ── Function as first-class value ────────────────────────────
def make_cost_calculator(price_per_1m_tokens: float) -> Callable[[int], float]:
    def calculate(tokens: int) -> float:
        return tokens / 1_000_000 * price_per_1m_tokens
    return calculate

gpt4o_cost  = make_cost_calculator(2.50)
claude_cost = make_cost_calculator(3.00)
print(f"\n5000 tokens — GPT-4o: ${gpt4o_cost(5000):.4f}  |  Claude: ${claude_cost(5000):.4f}")

## A7 — Comprehensions: List, Dict, Set & Generator Expressions

### 📚 Textbook Definition
Comprehensions are concise expressions for creating collections. Generator expressions use `()` and are **lazy** — they produce one element at a time with O(1) memory.

### 🎯 Interview Takeaway
List comprehension: `[x for x in data if cond]` vs generator: `(x for x in data if cond)`. In GenAI pipelines always prefer generators for large corpora.

In [ ]:
# ── List comprehension ───────────────────────────────────────
scores = [0.95, 0.42, 0.87, 0.31, 0.78]
relevant_scores = [s for s in scores if s > 0.6]
print("Relevant scores:", relevant_scores)

# ── Dict comprehension for model pricing index ────────────────
models_pricing = [
    ("gpt-4o", 2.50), ("gpt-4o-mini", 0.15), ("claude-3-5-sonnet", 3.00)
]
price_index = {name: price for name, price in models_pricing}
print("Price index:", price_index)

# ── Set comprehension for unique categories ───────────────────
chunk_tags = [["rag", "embedding"], ["rag", "llm"], ["llm", "agent"]]
all_tags = {tag for group in chunk_tags for tag in group}
print("Unique tags:", all_tags)

# ── Nested list comprehension: build message batch ───────────
prompts   = ["Explain RAG", "Explain LoRA"]
models    = ["gpt-4o-mini", "claude-haiku"]
batch     = [(p, m) for p in prompts for m in models]
print("\nCross-product batch:", batch)

# ── Generator expression: memory-efficient token counter ─────
documents = [f"Document {i}: {'word ' * (i*10+1)}" for i in range(1000)]
total_tokens = sum(math.ceil(len(d) / 4) for d in documents)   # generator, O(1) memory
print(f"\nApprox tokens across 1000 docs: {total_tokens:,}")

## 🎯 Section A — MAANG Interview Preparation

### Core Conceptual Questions

**Q: Why is a Python list `O(1)` for indexed access but `O(N)` for search?**
- ❌ *Beginner*: "Lists are fast."
- ✅ *Strong Engineer*: Lists store contiguous pointers to objects. Index access computes the pointer offset directly (`base + i * pointer_size`) in constant time. Linear search (`in list`) scans each pointer sequentially — O(N).

**Q: Why does `dict.get(key, default)` outperform `try/except KeyError` on hot paths?**
- ✅ *Strong Engineer*: `.get()` is a single C-level hash table lookup returning a default if the slot is empty. `try/except` constructs an exception object on the miss path (allocating a C struct), which costs roughly 10–50× more on the miss case.

**Q: When would you use `(str, Enum)` vs plain `Enum` in a model routing table?**
- ✅ *Strong Engineer*: `(str, Enum)` members *are* strings — they serialize directly in `json.dumps`, compare with `==` against string literals, and are accepted natively by FastAPI query parameters. Plain `Enum` members require `.value` extraction before serialization.

### Coding Challenges

In [ ]:
# ── Challenge 1: Order-preserving deduplication ─────────────
def deduplicate(items: list) -> list:
    seen = set()
    return [x for x in items if not (x in seen or seen.add(x))]

print("Deduped:", deduplicate(["a", "b", "a", "c", "b"]))

# ── Challenge 2: Safe deep-path getter ───────────────────────
def deep_get(obj: dict, *keys, default=None):
    for k in keys:
        if not isinstance(obj, dict):
            return default
        obj = obj.get(k)
    return obj if obj is not None else default

resp = {"choices": [{"message": {"content": "Hello"}}]}
print("Deep get:", deep_get(resp, "choices", 0, "message", "content"))
print("Missing: ", deep_get(resp, "choices", 0, "tool_calls", default=[]))

# ── Challenge 3: Flatten nested list of token arrays ─────────
nested = [[1, 2], [3, 4, 5], [6]]
flat   = [tok for seq in nested for tok in seq]
print("Flattened tokens:", flat)

---
# 🧠 SECTION B — PYTHON INTERNALS
*Object Identity, Memory Model, Reference Counting, Garbage Collection, GIL*

## B1 — Object Identity: `==` vs `is`, `id()`, Interning

### 📚 Textbook Definition
Every Python object has a unique **identity** (`id()`), a **type**, and a **value**. `is` tests identity (same object). `==` tests value equality (`__eq__`).

### 💀 Common Mistake
Using `is` for string or integer comparisons — CPython's interning makes it *sometimes* work, causing subtle bugs on dynamically constructed strings.

In [ ]:
# ── Identity vs Equality ─────────────────────────────────────
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print(f"a == b : {a == b}   (same values)")
print(f"a is b : {a is b}   (different objects)")
print(f"a is c : {a is c}   (same object!)")
print(f"id(a)={id(a)}, id(b)={id(b)}, id(c)={id(c)}")

# ── None, True, False are singletons ─────────────────────────
x = None
print(f"\nx is None : {x is None}")   # ALWAYS correct
print(f"x == None : {x == None}")     # Works but bad style

# ── String interning trap ─────────────────────────────────────
s1 = "model"
s2 = "model"
s3 = "".join(["m", "o", "d", "e", "l"])   # dynamically built

print(f"\ns1 is s2 : {s1 is s2}   (interned literal)")
print(f"s1 is s3 : {s1 is s3}   (dynamic string — may differ)")
print(f"s1 == s3 : {s1 == s3}   (always correct equality check)")

# ── Integer caching (-5 to 256) ──────────────────────────────
i1, i2 = 256, 256
i3, i4 = 257, 257
print(f"\n256 is 256 : {i1 is i2}")   # True (cached)
print(f"257 is 257 : {i3 is i4}")     # False (not cached)

## B2 — Reference Counting, Shallow vs Deep Copy

### 📚 Textbook Definition
CPython tracks how many names reference an object via a **reference count**. When it reaches zero the object is deallocated. `copy.copy()` creates a new container but shares child object references. `copy.deepcopy()` recursively clones the entire object graph.

### 🎯 Interview Takeaway
For nested agent states or model configs, **always use `deepcopy`** unless you explicitly intend to share child objects.

In [ ]:
import sys, copy

# ── Reference counting ────────────────────────────────────────
cfg = {"temperature": 0.7, "model": "gpt-4o"}
ref_a = cfg          # ref count +1
ref_b = cfg          # ref count +1
# sys.getrefcount returns count+1 (because argument itself is a ref)
print(f"Reference count: {sys.getrefcount(cfg) - 1}")

# ── Shallow copy danger ───────────────────────────────────────
base  = {"name": "agent", "params": {"temperature": 0.7}}
shallow = copy.copy(base)
shallow["params"]["temperature"] = 0.0   # MUTATES base!
print(f"Base params after shallow copy mutation: {base['params']}")

# ── Deep copy isolation ───────────────────────────────────────
base["params"]["temperature"] = 0.7      # reset
deep = copy.deepcopy(base)
deep["params"]["temperature"] = 0.0     # isolated
print(f"Base params after deep copy mutation  : {base['params']}")
print(f"Deep copy params                       : {deep['params']}")

# ── When NOT to use deepcopy: Performance ────────────────────
import time

large = [{"id": i, "vec": list(range(128))} for i in range(1000)]
t0 = time.perf_counter()
_ = [d.copy() for d in large]      # shallow copies of top-level dicts
t1 = time.perf_counter()
_ = copy.deepcopy(large)           # full recursive copy
t2 = time.perf_counter()

print(f"\nShallow (1k records): {(t1-t0)*1000:.2f}ms")
print(f"Deep    (1k records): {(t2-t1)*1000:.2f}ms  ({(t2-t1)/(t1-t0):.0f}x slower)")

## B3 — Memory Model: Mutability, Assignment & the GIL

### 📚 Textbook Definition
**Mutable** objects (`list`, `dict`, `set`) can be modified in place. **Immutable** objects (`int`, `str`, `tuple`, `frozenset`) cannot. The **GIL** (Global Interpreter Lock) prevents true CPU parallelism in CPython for pure-Python threads — threads still yield speedups for **I/O-bound** tasks.

### 🧠 Intuition
The GIL is a single master key to the CPython interpreter. Only one thread holds it at a time. During I/O waits (network calls, disk reads), the GIL is released — so 100 threads hitting an LLM API each run at 100% efficiency.

In [ ]:
# ── Mutability demonstration ─────────────────────────────────
# Immutable int: reassignment creates NEW object
x = 10
old_id = id(x)
x += 1
print(f"Int reassigned: id changed = {id(x) != old_id}")

# Mutable list: in-place modification, same object
lst = [1, 2, 3]
old_id = id(lst)
lst.append(4)
print(f"List appended:  id unchanged = {id(lst) == old_id}")

# ── Tuple as immutable record ─────────────────────────────────
dim_spec = (768, "float32")    # embedding shape — should not change
try:
    dim_spec[0] = 1024         # type: ignore
except TypeError as e:
    print(f"Tuple mutation blocked: {e}")

# ── GIL impact illustration ───────────────────────────────────
import threading

counter = 0
def increment(n: int):
    global counter
    for _ in range(n):
        counter += 1  # NOT thread-safe! GIL does NOT protect compound ops

threads = [threading.Thread(target=increment, args=(10_000,)) for _ in range(4)]
[t.start() for t in threads]; [t.join() for t in threads]
print(f"\nExpected counter: 40000 | Actual: {counter}")
print("(Race condition — GIL doesn't protect multi-step read-modify-write!)")

## B4 — Garbage Collection & Memory Leaks in GenAI Applications

### 📚 Textbook Definition
CPython uses **reference counting** as its primary GC mechanism, supplemented by a **cyclic garbage collector** for detecting reference cycles (e.g. A references B, B references A).

### 💀 Common Mistake in GenAI Systems
Accumulating all chat history turns in a global in-memory list without bound — a RAM leak that causes OOM crashes after long-running sessions.

In [ ]:
import gc

# ── Reference cycle ────────────────────────────────────────────
class Node:
    def __init__(self, name):
        self.name = name
        self.ref = None   # Will create a cycle

a = Node("A")
b = Node("B")
a.ref = b   # A -> B
b.ref = a   # B -> A  (cycle!)

# Drop names — reference count doesn't reach zero due to cycle
a_id, b_id = id(a), id(b)
del a, b

collected = gc.collect()   # Cyclic GC detects and collects the cycle
print(f"Cyclic GC collected {collected} objects from reference cycle")

# ── Bounded conversation buffer (prevent memory leak) ─────────
class BoundedConversationBuffer:
    def __init__(self, max_turns: int = 20):
        self.max_turns = max_turns
        self._turns: list = []

    def add_turn(self, role: str, content: str):
        self._turns.append({"role": role, "content": content})
        # Sliding window — evict oldest non-system turns
        user_assistant = [t for t in self._turns if t["role"] != "system"]
        system_turns   = [t for t in self._turns if t["role"] == "system"]
        if len(user_assistant) > self.max_turns * 2:
            user_assistant = user_assistant[-self.max_turns * 2:]
        self._turns = system_turns + user_assistant

    def get_messages(self) -> list:
        return self._turns.copy()

buf = BoundedConversationBuffer(max_turns=2)
for i in range(10):
    buf.add_turn("user", f"Turn {i} question")
    buf.add_turn("assistant", f"Turn {i} answer")

print(f"Buffer size (max 4+sys): {len(buf.get_messages())} turns")

## 🎯 Section B — MAANG Interview Prep

### Interview Questions

**Q: What is the GIL and when does it matter for GenAI applications?**
- ✅ *Strong Engineer*: The GIL serializes CPython bytecode execution across threads. For **I/O-bound** LLM API calls, the GIL is released during socket waits — making threading and asyncio fully effective. For **CPU-bound** tokenization or embedding post-processing, the GIL blocks true parallelism — requiring `multiprocessing` or native Cython/C extensions.

**Q: What is the difference between `del x` and `x = None` in Python?**
- ✅ *Strong Engineer*: `del x` removes the name binding from the local namespace, decrementing the object's reference count. `x = None` rebinds the name to the `None` singleton, also decrementing the previous object's refcount. Both trigger deallocation if the refcount reaches zero — but `del x` also removes the name itself.

In [ ]:
# ── Memory sizing in GenAI context ──────────────────────────
import sys

# Python float (C double wrapper)
f = 3.14
print(f"Python float size:    {sys.getsizeof(f)} bytes")  # ~24 bytes!

# Compare: in a numpy array, float32 = 4 bytes, float64 = 8 bytes
# For 1M-dim embedding space at 768 dims:
n_vectors = 10_000
dim       = 768
py_list_mb    = (n_vectors * dim * sys.getsizeof(0.0)) / 1e6
numpy_f32_mb  = (n_vectors * dim * 4) / 1e6   # float32
print(f"\n10k x 768 embeddings:")
print(f"  Python list of floats: ~{py_list_mb:.0f} MB")
print(f"  NumPy float32 array  : ~{numpy_f32_mb:.0f} MB  ({py_list_mb/numpy_f32_mb:.0f}x smaller)")

---
# ⚙️ SECTION C — ADVANCED PYTHON
*Decorators, Iterables, Iterators, Generators, Context Managers, Exception Handling, Dataclasses, Type Hints, OOP, Factory Pattern, Pydantic*

## C1 — Decorators & functools.wraps

### 📚 Textbook Definition
A decorator is a **higher-order function** that wraps another function to augment its behaviour without modifying it. `@functools.wraps(func)` copies the original function's metadata to the wrapper.

### 💀 Common Mistake
Omitting `@functools.wraps` — this destroys `__name__`, `__doc__`, and `__annotations__`, breaking FastAPI route discovery and debugging stack traces.

In [ ]:
import functools, time, random

# ── 1. Timing decorator ──────────────────────────────────────
def timeit(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f"⏱  {func.__name__} took {elapsed_ms:.2f}ms")
        return result
    return wrapper

# ── 2. Parametrised decorator: retry with jitter ────────────
def retry(max_attempts: int = 3, base_delay: float = 0.05,
          retryable: tuple = (ConnectionError, TimeoutError)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except retryable as e:
                    if attempt == max_attempts:
                        raise
                    wait = random.uniform(0.5, 1.5) * base_delay * (2 ** (attempt - 1))
                    print(f"  ↻ Attempt {attempt} failed ({type(e).__name__}). Retry in {wait:.3f}s")
                    time.sleep(wait)
        return wrapper
    return decorator

# ── Demo ──────────────────────────────────────────────────────
call_count = 0

@timeit
@retry(max_attempts=3, base_delay=0.01, retryable=(ConnectionError,))
def flaky_llm_call(prompt: str) -> str:
    """Simulates a flaky LLM API endpoint."""
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"HTTP 503 Service Unavailable")
    return f"Response to: '{prompt}'"

result = flaky_llm_call("Explain attention")
print("Result:", result)
print("functools.wraps preserved __doc__:", flaky_llm_call.__doc__)

## C2 — Iterables, Iterators & Generators

### 📚 Textbook Definition
- **Iterable**: has `__iter__()`. Returns an iterator.
- **Iterator**: has `__next__()`. Raises `StopIteration` when exhausted.
- **Generator**: a function using `yield`. Automatically creates an iterator with O(1) memory.

### 🎯 Interview Takeaway
A generator is a **lazy iterator** — it computes values on-demand. For streaming LLM responses or 100M document corpora, generators eliminate OOM crashes that lists would cause.

In [ ]:
import sys

# ── 1. Custom Iterator class ──────────────────────────────────
class TokenBatcher:
    """Yields fixed-size token batches from a flat token list."""
    def __init__(self, tokens: list, batch_size: int):
        self.tokens = tokens
        self.batch_size = batch_size
        self._index = 0

    def __iter__(self): return self

    def __next__(self) -> list:
        if self._index >= len(self.tokens):
            raise StopIteration
        batch = self.tokens[self._index:self._index + self.batch_size]
        self._index += self.batch_size
        return batch

tokens = list(range(10))
for batch in TokenBatcher(tokens, batch_size=3):
    print("Batch:", batch)

# ── 2. Generator function: streaming tokens ───────────────────
def stream_completion(sentence: str, chunk_size: int = 3) -> Generator:
    """Simulates character-by-character token streaming."""
    words = sentence.split()
    for i in range(0, len(words), chunk_size):
        yield " ".join(words[i:i+chunk_size])

stream = stream_completion("Retrieval Augmented Generation enhances LLMs by grounding them in facts", 2)
print("\nStreamed chunks:")
for chunk in stream:
    print(f"  chunk: '{chunk}'")

# ── 3. Memory: generator vs list ──────────────────────────────
N = 100_000
gen  = (f"Document {i}" for i in range(N))
lst  = [f"Document {i}" for i in range(N)]
print(f"\nGenerator container size : {sys.getsizeof(gen):>8} bytes")
print(f"List container size      : {sys.getsizeof(lst):>8} bytes  ({sys.getsizeof(lst)//sys.getsizeof(gen)}x larger)")
del lst   # free memory

# ── 4. Pipeline chaining ──────────────────────────────────────
def load_docs(n): return (f"Doc {i}: content chunk here." for i in range(n))
def chunk(docs):  return (sentence for doc in docs for sentence in doc.split(".") if sentence.strip())
def embed(chunks): return ({"text": c.strip(), "vec": [hash(c) % 100 / 100]} for c in chunks)

pipeline = embed(chunk(load_docs(3)))
print("\nPipeline output (first 3):")
for item in list(pipeline)[:3]:
    print(" ", item)

## C3 — Context Managers: `with`, `__enter__/__exit__`, `contextlib`

### 📚 Textbook Definition
A context manager implements `__enter__()` (setup) and `__exit__()` (teardown). The `with` statement guarantees teardown even if an exception is raised.

### 💀 Common Mistake
Using `file.close()` manually — if an exception occurs before that line, the file descriptor leaks permanently.

In [ ]:
from contextlib import contextmanager

# ── 1. Class-based context manager ───────────────────────────
class LLMSpanTracer:
    """Tracks latency and token usage for a named LLM operation."""
    def __init__(self, span_name: str):
        self.span_name = span_name
        self.start = 0.0

    def __enter__(self):
        self.start = time.perf_counter()
        print(f"▶ [{self.span_name}] started")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = (time.perf_counter() - self.start) * 1000
        status = "FAILED" if exc_type else "OK"
        print(f"■ [{self.span_name}] {status} in {elapsed:.2f}ms")
        return False  # propagate exceptions

with LLMSpanTracer("RAG retrieval"):
    time.sleep(0.02)

# ── 2. Generator-based context manager ────────────────────────
@contextmanager
def temp_system_prompt(agent: dict, override: str):
    """Temporarily overrides system prompt, restoring original on exit."""
    original = agent.get("system_prompt")
    agent["system_prompt"] = override
    try:
        yield agent
    finally:
        agent["system_prompt"] = original

agent_state = {"system_prompt": "Standard assistant", "mode": "default"}
print("\nBefore:", agent_state["system_prompt"])
with temp_system_prompt(agent_state, "RED TEAM AUDITOR — test adversarial prompts") as a:
    print("Inside:", a["system_prompt"])
print("After: ", agent_state["system_prompt"])

# ── 3. Context manager for file write ────────────────────────
@contextmanager
def atomic_json_write(path: str):
    """Writes JSON atomically — temp file swapped on success."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        yield f
    import os; os.replace(tmp_path, path)
    print(f"Atomically written to {path}")

with atomic_json_write("/tmp/genai_output.json") as f:
    import json; json.dump({"status": "ok", "cost": 0.0012}, f)

## C4 — Exception Handling: Custom Exceptions & Exception Chaining

### 📚 Textbook Definition
Custom exceptions subclass `Exception` to create domain-specific error types. Exception chaining (`raise X from Y`) preserves the original traceback as `__cause__`.

In [ ]:
# ── Domain exception hierarchy ───────────────────────────────
class GenAIPlatformError(Exception):
    """Root error for all platform failures."""

class ProviderRateLimitError(GenAIPlatformError):
    def __init__(self, provider: str, retry_after: int = 5):
        super().__init__(f"{provider} rate limit hit. Retry after {retry_after}s")
        self.retry_after = retry_after

class ContextWindowExceededError(GenAIPlatformError):
    def __init__(self, tokens: int, limit: int):
        super().__init__(f"Prompt has {tokens} tokens; model limit is {limit}")
        self.tokens, self.limit = tokens, limit

class ToolExecutionError(GenAIPlatformError):
    def __init__(self, tool: str, reason: str):
        super().__init__(f"Tool '{tool}' failed: {reason}")

# ── try/except/else/finally pattern ──────────────────────────
def execute_llm_turn(tokens: int) -> str:
    try:
        if tokens > 8192:
            raise ContextWindowExceededError(tokens, 8192)
        result = f"Generated answer ({tokens} tokens)"
    except ContextWindowExceededError as cwe:
        print(f"⚠ Context overflow: {cwe}. Routing to 128k model.")
        result = "Routed to extended context model."
    except ProviderRateLimitError as rle:
        print(f"⚠ Rate limit: retry in {rle.retry_after}s")
        result = "Queued for retry."
    else:
        print(f"✅ Generation successful: {result}")
    finally:
        print(f"  [Telemetry recorded: tokens={tokens}]")
    return result

execute_llm_turn(512)
print()
execute_llm_turn(10000)

# ── Exception chaining ────────────────────────────────────────
import json
try:
    try:
        json.loads("{{invalid json}}")
    except json.JSONDecodeError as raw:
        raise ToolExecutionError("json_parser", "Model output was not valid JSON") from raw
except ToolExecutionError as te:
    print(f"\nCaught: {te}")
    print(f"Root cause: {te.__cause__}")

## C5 — Dataclasses, Type Hints & Enums

### 📚 Textbook Definition
`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__`. `frozen=True` makes instances immutable. `field(default_factory=...)` safely handles mutable defaults.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional

class ModelTier(str, Enum):
    FAST      = "fast"
    STANDARD  = "standard"
    FRONTIER  = "frontier"

class FinishReason(str, Enum):
    STOP          = "stop"
    LENGTH        = "length"
    TOOL_CALL     = "tool_calls"
    CONTENT_FILTER = "content_filter"

@dataclass(frozen=True)
class ModelSpec:
    model_id: str
    tier: ModelTier
    context_window: int
    cost_prompt_per_1m: float
    cost_completion_per_1m: float

    def estimated_cost(self, prompt_tok: int, completion_tok: int) -> float:
        return (prompt_tok / 1e6 * self.cost_prompt_per_1m +
                completion_tok / 1e6 * self.cost_completion_per_1m)

@dataclass
class Chunk:
    chunk_id: str
    text: str
    embedding: Optional[list] = None
    tags: list = field(default_factory=list)      # SAFE mutable default

    def __post_init__(self):
        if not self.text.strip():
            raise ValueError(f"Chunk {self.chunk_id} has empty text!")

# Usage
gpt4o = ModelSpec("gpt-4o", ModelTier.FRONTIER, 128_000, 2.50, 10.00)
print("Model:", gpt4o)
print(f"Cost for 1k+500 tokens: ${gpt4o.estimated_cost(1000, 500):.4f}")

chunk = Chunk("c001", "RAG grounds LLM answers in facts.", tags=["rag"])
print("\nChunk:", chunk)

try:
    Chunk("c002", "   ")  # Should raise
except ValueError as e:
    print(f"Validation caught: {e}")

## C6 — Object-Oriented Python: ABCs & Factory Pattern

### 📚 Textbook Definition
An **Abstract Base Class** (ABC) enforces method implementation contracts. The **Factory Pattern** decouples object creation from usage, enabling runtime provider selection.

### 🎯 Interview Takeaway
ABCs prevent silent `AttributeError` failures in production — if `generate()` is abstract, Python refuses to instantiate a concrete class that doesn't implement it.

In [ ]:
from abc import ABC, abstractmethod

# ── Abstract provider interface ───────────────────────────────
class LLMProvider(ABC):
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key

    @abstractmethod
    def generate(self, messages: list, **kwargs) -> str: ...

    @abstractmethod
    def embed(self, text: str) -> list: ...

    @property
    @abstractmethod
    def provider_name(self) -> str: ...

# ── Concrete implementations ──────────────────────────────────
class OpenAIProvider(LLMProvider):
    @property
    def provider_name(self) -> str: return "openai"
    def generate(self, messages, **kwargs) -> str:
        return f"[OpenAI] Response to: {messages[-1]['content']}"
    def embed(self, text: str) -> list:
        return [hash(w) % 1000 / 1000 for w in text.split()[:4]]

class AnthropicProvider(LLMProvider):
    @property
    def provider_name(self) -> str: return "anthropic"
    def generate(self, messages, **kwargs) -> str:
        return f"[Anthropic] Response to: {messages[-1]['content']}"
    def embed(self, text: str) -> list:
        return [abs(hash(w)) % 1000 / 1000 for w in text.split()[:4]]

# ── Factory ────────────────────────────────────────────────────
class LLMFactory:
    _registry: dict = {}

    @classmethod
    def register(cls, name: str):
        def decorator(klass):
            cls._registry[name] = klass
            return klass
        return decorator

    @classmethod
    def create(cls, name: str, **kwargs) -> LLMProvider:
        if name not in cls._registry:
            raise ValueError(f"Unknown provider '{name}'. Available: {list(cls._registry)}")
        return cls._registry[name](**kwargs)

LLMFactory._registry["openai"]    = OpenAIProvider
LLMFactory._registry["anthropic"] = AnthropicProvider

# Polymorphic dispatch — caller doesn't know which provider!
for provider_name in ["openai", "anthropic"]:
    provider = LLMFactory.create(provider_name)
    response = provider.generate([{"role": "user", "content": "What is RAG?"}])
    print(f"[{provider.provider_name}]: {response}")

## C7 — Pydantic v2: Validation, Schemas & Tool Definitions

### 📚 Textbook Definition
Pydantic v2 (Rust core) parses and validates data against Python type annotations. `model_json_schema()` generates JSON Schema draft-07 — the exact format LLMs need for structured output and tool calling.

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError
from typing import Literal

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant", "tool"]
    content: str = Field(..., min_length=1)

class GenerationConfig(BaseModel):
    model: str = Field(..., description="Model identifier")
    temperature: float = Field(0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(1024, ge=1, le=128_000)
    messages: list[ChatMessage] = Field(..., min_length=1)

    @field_validator("model")
    @classmethod
    def validate_model(cls, v):
        allowed = {"gpt-4o", "gpt-4o-mini", "claude-3-5-sonnet", "o1-mini"}
        if v not in allowed:
            raise ValueError(f"Model '{v}' not supported. Choose from {allowed}")
        return v

    @model_validator(mode="after")
    def reasoning_model_temp(self):
        if self.model.startswith("o1") and self.temperature != 1.0:
            object.__setattr__(self, "temperature", 1.0)
        return self

# Valid request
cfg = GenerationConfig(
    model="gpt-4o",
    temperature=0.2,
    messages=[ChatMessage(role="user", content="Explain transformers")]
)
print("Config:", cfg.model_dump_json(indent=2))

# Invalid — caught pre-flight
try:
    GenerationConfig(model="palm-2", temperature=3.5,
                     messages=[ChatMessage(role="user", content="hi")])
except ValidationError as e:
    print("\nValidation errors:")
    for err in e.errors():
        print(f"  [{err['loc'][0]}]: {err['msg']}")

# Tool schema generation
class WebSearchTool(BaseModel):
    """Search the web and return the top results."""
    query: str = Field(..., description="Search query string")
    num_results: int = Field(5, ge=1, le=20, description="Number of results to return")
    language: str = Field("en", description="ISO language code")

import json
print("\nTool JSON Schema:")
print(json.dumps(WebSearchTool.model_json_schema(), indent=2))

## 🎯 Section C — MAANG Interview Prep

**Q: Why is `@functools.wraps` non-negotiable in production decorators?**
> FastAPI uses `__name__` and `__annotations__` for route registration and OpenAPI schema generation. Omitting `@functools.wraps` causes all decorated routes to appear as the wrapper function name — breaking auto-generated docs and test discovery.

**Q: When does a generator outperform a list in a RAG embedding pipeline?**
> A list comprehension eagerly materialises all N document chunks in RAM before the first embedding call. A generator yields one chunk at a time — the embedding model processes it immediately and the chunk is discarded. For 10M document corpora this is the difference between 40GB RAM and 40MB.

**Q: What is the difference between `@classmethod` and `@staticmethod`?**
> `@classmethod` receives the class (`cls`) as the first argument — allowing subclass-aware operations like factory methods. `@staticmethod` receives no implicit argument — it's a plain function namespaced inside the class, appropriate for utility helpers with no class or instance state access.

In [ ]:
# ── Coding challenge: composable decorator stack ─────────────
def validate_prompt(min_length=5):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(prompt: str, *args, **kwargs):
            if len(prompt.strip()) < min_length:
                raise ValueError(f"Prompt too short (min {min_length} chars)")
            return func(prompt, *args, **kwargs)
        return wrapper
    return decorator

def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
@timeit
@validate_prompt(min_length=5)
def call_model(prompt: str) -> str:
    time.sleep(0.01)
    return f"Answer to: {prompt}"

print(call_model("Explain embeddings"))
print(f"Total calls: {call_model.calls}")
try:
    call_model("Hi")
except ValueError as e:
    print(f"Blocked: {e}")

---
# ⚡ SECTION D — CONCURRENCY
*Threading, Multiprocessing, asyncio, async/await, Event Loops, Locks, Race Conditions, Thread Pools, Process Pools*

## D1 — Concurrency vs Parallelism & the Python Concurrency Model

### 📚 Textbook Definition
- **Concurrency**: Managing multiple tasks by interleaving (single core, cooperative).
- **Parallelism**: Executing multiple tasks simultaneously (multiple cores).
- **GIL**: Limits CPython to one thread executing Python bytecode at a time.

| Workload Type | Use | Why |
|---|---|---|
| I/O-bound (LLM APIs) | `asyncio` / `threading` | GIL released during I/O waits |
| CPU-bound (tokenization, inference) | `multiprocessing` | Bypasses GIL via separate processes |
| Mixed | `asyncio` + `ProcessPoolExecutor` | Best of both worlds |

In [ ]:
import threading, time, concurrent.futures

# ── Sequential vs Threaded I/O benchmark ─────────────────────
def simulated_api_call(task_id: int, latency: float = 0.05) -> str:
    time.sleep(latency)   # Simulates network I/O (GIL released during sleep)
    return f"Result_{task_id}"

N = 10

# Sequential
t0 = time.perf_counter()
results_seq = [simulated_api_call(i) for i in range(N)]
t_seq = time.perf_counter() - t0

# Threaded (I/O-bound — GIL doesn't hurt)
t1 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=N) as pool:
    results_thr = list(pool.map(simulated_api_call, range(N)))
t_thr = time.perf_counter() - t1

print(f"Sequential : {t_seq:.3f}s")
print(f"Threaded   : {t_thr:.3f}s  ({t_seq/t_thr:.1f}x faster)")
print(f"All results match: {results_seq == results_thr}")

## D2 — Threading: Locks, Race Conditions & Thread-Safe Design

### 📚 Textbook Definition
A **Race Condition** occurs when multiple threads read-modify-write shared state concurrently. A `threading.Lock` (mutex) enforces mutual exclusion — only one thread holds it at a time.

### 💀 Common Mistake
Using a plain `int` counter as a shared budget tracker without a Lock — leads to incorrect, non-deterministic values.

In [ ]:
import threading

# ── Race condition demonstration ──────────────────────────────
unsafe_balance = 0
safe_balance   = 0
lock = threading.Lock()

def unsafe_deposit(amount: int):
    global unsafe_balance
    current = unsafe_balance
    time.sleep(0.0001)      # Force interleaving
    unsafe_balance = current + amount

def safe_deposit(amount: int):
    global safe_balance
    with lock:
        safe_balance += amount   # Atomic under lock

N_THREADS = 50
threads_unsafe = [threading.Thread(target=unsafe_deposit, args=(10,)) for _ in range(N_THREADS)]
threads_safe   = [threading.Thread(target=safe_deposit,   args=(10,)) for _ in range(N_THREADS)]

for t in threads_unsafe: t.start()
for t in threads_unsafe: t.join()

for t in threads_safe: t.start()
for t in threads_safe: t.join()

expected = N_THREADS * 10
print(f"Expected balance : {expected}")
print(f"Unsafe balance   : {unsafe_balance}  ({'✅' if unsafe_balance == expected else '❌ RACE CONDITION'})")
print(f"Safe balance     : {safe_balance}    ({'✅' if safe_balance   == expected else '❌'})")

# ── Thread-safe token budget tracker ──────────────────────────
class ThreadSafeTokenBudget:
    def __init__(self, budget: int):
        self._budget = budget
        self._consumed = 0
        self._lock = threading.Lock()

    def consume(self, tokens: int) -> bool:
        with self._lock:
            if self._consumed + tokens > self._budget:
                return False
            self._consumed += tokens
            return True

    @property
    def remaining(self): return self._budget - self._consumed

budget = ThreadSafeTokenBudget(1000)
def worker():
    return budget.consume(150)

threads = [threading.Thread(target=worker) for _ in range(8)]
[t.start() for t in threads]; [t.join() for t in threads]
print(f"\nBudget remaining: {budget.remaining} (6 of 8 threads approved at 150 tokens each)")

## D3 — asyncio: Event Loop, async/await, gather & Semaphore

### 📚 Textbook Definition
`asyncio` provides a **cooperative multitasking** event loop on a single thread. `await` yields control to the loop when waiting for I/O. `asyncio.gather()` schedules multiple coroutines concurrently. `asyncio.Semaphore(N)` limits max concurrent tasks to N.

### 💀 Common Mistake
Calling `time.sleep()` or `requests.get()` inside an `async def` — these block the entire event loop, preventing ALL other coroutines from running.

In [ ]:
import asyncio

# ── 1. Basic coroutine comparison ────────────────────────────
async def mock_embed(text: str, delay: float = 0.03) -> list:
    await asyncio.sleep(delay)   # Non-blocking I/O wait
    return [hash(w) % 100 / 100 for w in text.split()[:4]]

# ── 2. 20x concurrent vs sequential benchmark ────────────────
async def benchmark():
    queries = [f"Query about topic {i}" for i in range(20)]

    # Sequential
    t0 = time.perf_counter()
    for q in queries:
        await mock_embed(q)
    t_seq = time.perf_counter() - t0

    # Concurrent gather
    t1 = time.perf_counter()
    await asyncio.gather(*(mock_embed(q) for q in queries))
    t_conc = time.perf_counter() - t1

    print(f"Sequential  (20 embeds): {t_seq:.3f}s")
    print(f"Concurrent  (20 embeds): {t_conc:.3f}s  ({t_seq/t_conc:.1f}x faster)")

await benchmark()

# ── 3. Semaphore: safe bounded concurrency ────────────────────
async def bounded_embed_pipeline(texts: list, max_concurrent: int = 3):
    sem = asyncio.Semaphore(max_concurrent)
    active = 0

    async def worker(text: str) -> dict:
        nonlocal active
        async with sem:
            active += 1
            result = await mock_embed(text)
            active -= 1
            return {"text": text, "embedding_dims": len(result)}

    return await asyncio.gather(*(worker(t) for t in texts))

texts = [f"Document {i}" for i in range(10)]
results = await bounded_embed_pipeline(texts, max_concurrent=3)
print(f"\nProcessed {len(results)} documents with max 3 concurrent slots")
print("Sample:", results[0])

# ── 4. Timeout & cancellation ────────────────────────────────
async def slow_llm(): await asyncio.sleep(5.0); return "slow answer"
async def fast_slm(): await asyncio.sleep(0.02); return "fast answer"

async def hedge_request():
    task_slow = asyncio.create_task(slow_llm())
    task_fast = asyncio.create_task(fast_slm())
    done, pending = await asyncio.wait([task_slow, task_fast],
                                       return_when=asyncio.FIRST_COMPLETED)
    winner = next(iter(done)).result()
    for p in pending: p.cancel()
    return winner

winner = await hedge_request()
print(f"\nHedged winner: '{winner}'")

## D4 — Multiprocessing & ProcessPoolExecutor

### 📚 Textbook Definition
`multiprocessing` spawns separate OS processes, each with its own memory space and Python interpreter — bypassing the GIL entirely. Ideal for CPU-intensive tasks like tokenization, BPE encoding, or local model inference.

### 🎯 Interview Takeaway
Pickle overhead (serialising arguments and return values) makes multiprocessing inefficient for tiny tasks. Use it when each task takes >50ms of pure CPU work.

In [ ]:
import concurrent.futures, math, multiprocessing, os

def cpu_heavy_tokenize(doc: str) -> dict:
    """Simulates CPU-intensive BPE tokenisation."""
    tokens = doc.lower().split()
    char_count = sum(len(t) for t in tokens)
    # Fake heavy work
    for _ in range(20_000): _ = math.sqrt(char_count)
    return {"doc": doc[:20], "token_count": len(tokens)}

docs = [f"Enterprise document with technical language number {i} " * 5 for i in range(8)]

# Single process
t0 = time.perf_counter()
results_single = [cpu_heavy_tokenize(d) for d in docs]
t_single = time.perf_counter() - t0

# ProcessPoolExecutor — bypasses GIL for CPU-bound work
# In notebooks/Unix, use fork context to serialize dynamically defined cell functions
ctx = multiprocessing.get_context("fork") if hasattr(os, "fork") else None
t1 = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(mp_context=ctx) as pool:
    results_multi = list(pool.map(cpu_heavy_tokenize, docs))
t_multi = time.perf_counter() - t1

print(f"Single process : {t_single:.3f}s")
print(f"ProcessPool    : {t_multi:.3f}s  ({t_single/max(t_multi,0.001):.1f}x)")
print(f"Results match  : {[r['doc'] for r in results_single] == [r['doc'] for r in results_multi]}")

## 🎯 Section D — MAANG Interview Prep

**Q: Why does `asyncio` outperform threading for LLM API calls despite being single-threaded?**
> Threading has overhead: context switches (~1–10μs), OS scheduler involvement, and lock contention. `asyncio` cooperatively yields at `await` points — context switching is pure Python function calls (~100ns). For 1000 concurrent I/O operations, `asyncio` creates 1000 coroutines on one thread; threading would require 1000 OS threads consuming ~8MB stack each.

**Q: How would you correctly process 1 million document chunks with both CPU-heavy tokenization AND network-bound embedding API calls?**
> Two-stage pipeline: Stage 1 uses `ProcessPoolExecutor` for CPU-bound tokenisation. Stage 2 uses `asyncio` + `Semaphore` for concurrent rate-limited embedding API calls, accepting tokenised chunks from a `queue.Queue`.

In [ ]:
# ── asyncio + thread pool (blocking code offloaded safely) ───
import asyncio

def blocking_json_parse(raw: str) -> dict:
    """Simulates a CPU-blocking JSON operation."""
    import json
    time.sleep(0.01)  # Simulates heavy parsing
    return json.loads(raw)

async def safe_parse_async(raw: str) -> dict:
    """Runs blocking code in thread pool, keeping event loop free."""
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(None, blocking_json_parse, raw)

payloads = ['{"id": %d, "val": "data"}' % i for i in range(10)]
parsed = await asyncio.gather(*(safe_parse_async(p) for p in payloads))
print(f"Safely parsed {len(parsed)} JSON payloads using run_in_executor")
print("Sample:", parsed[0])

# 🌐 SECTION E — BACKEND ENGINEERING FOR GENAI

Every production LLM feature lives inside a backend system. Understanding HTTP protocols, FastAPI architecture, authentication, rate limiting, and queueing is what separates a notebook experimenter from an AI systems architect.

---

## E1 — HTTP, REST & The GenAI Wire Protocol

### 📚 Textbook Definition
REST over HTTP/1.1 and HTTP/2 is the transport layer for LLM APIs. Understanding status codes (`429 Too Many Requests`, `503 Service Unavailable`, `400 Bad Request`, `401 Unauthorized`), headers (`Authorization: Bearer <key>`, `Content-Type: text/event-stream`), and payloads is mandatory.

### 🧠 Intuition
When calling OpenAI or Anthropic, you are sending a standard `POST` request with a JSON body and streaming back an SSE (Server-Sent Events) chunked response.

In [ ]:
import json
from dataclasses import dataclass
from typing import Dict, Any, Optional

@dataclass
class MockHttpResponse:
    status_code: int
    headers: Dict[str, str]
    body: str
    
    def json(self) -> Any:
        return json.loads(self.body)

def simulate_llm_gateway(method: str, path: str, headers: dict, body: dict) -> MockHttpResponse:
    """Simulates an enterprise API Gateway with validation and routing."""
    if "Authorization" not in headers or not headers["Authorization"].startswith("Bearer "):
        return MockHttpResponse(401, {"Content-Type": "application/json"}, '{"error": "Unauthorized: Missing Bearer Token"}')
    
    token = headers["Authorization"].split(" ")[1]
    if token != "secret-genai-token":
        return MockHttpResponse(403, {"Content-Type": "application/json"}, '{"error": "Forbidden: Invalid Token"}')
        
    if method == "POST" and path == "/v1/chat/completions":
        if "messages" not in body or not isinstance(body["messages"], list):
            return MockHttpResponse(400, {"Content-Type": "application/json"}, '{"error": "Bad Request: messages list required"}')
        
        # Success response
        resp = {
            "id": "chatcmpl-mock-99",
            "model": body.get("model", "gpt-4o-mini"),
            "choices": [{"message": {"role": "assistant", "content": "Hello! I am your AI assistant."}}],
            "usage": {"prompt_tokens": 12, "completion_tokens": 9, "total_tokens": 21}
        }
        return MockHttpResponse(200, {"Content-Type": "application/json"}, json.dumps(resp))
        
    return MockHttpResponse(404, {"Content-Type": "application/json"}, '{"error": "Route Not Found"}')

# Test wire protocol simulation
auth_headers = {"Authorization": "Bearer secret-genai-token", "Content-Type": "application/json"}
res = simulate_llm_gateway("POST", "/v1/chat/completions", auth_headers, {"model": "gpt-4o", "messages": [{"role": "user", "content": "Hi"}]})
print("Status Code:", res.status_code)
print("Response JSON:", res.json())

## E2 — FastAPI Architecture & Dependency Injection (`Depends`)

### 📚 Textbook Definition
FastAPI uses Python type hints and ASGI (`Starlette`) to provide asynchronous HTTP handling, automatic OpenAPI schema generation, and a powerful hierarchical Dependency Injection system (`Depends`).

### 🧠 Intuition
In a GenAI backend, dependencies allow you to share rate-limiters, API key validators, semantic caches, and vector DB clients cleanly across multiple route handlers without global state.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header
from pydantic import BaseModel, Field
from typing import List, Optional
import asyncio

app = FastAPI(title="GenAI Inference Service", version="1.0.0")

# Request / Response DTOs
class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = "gpt-4o-mini"
    messages: List[Message]
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

class ChatResponse(BaseModel):
    reply: str
    tokens_used: int
    cached: bool = False

# Dependency: API Key Auth
def verify_api_key(x_api_key: str = Header(default="demo-key")):
    if x_api_key != "demo-key":
        raise HTTPException(status_code=401, detail="Invalid API Key")
    return {"user_id": "tenant-abc-123", "tier": "enterprise"}

# Endpoint using Dependency
@app.post("/chat", response_model=ChatResponse)
async def chat_endpoint(req: ChatRequest, user: dict = Depends(verify_api_key)):
    await asyncio.sleep(0.01) # Simulated model call
    last_msg = req.messages[-1].content
    return ChatResponse(
        reply=f"[Tenant: {user['user_id']}] Echo: {last_msg}",
        tokens_used=len(last_msg.split()) * 2
    )

print("FastAPI Application configured successfully with routes:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"  {list(route.methods)} {route.path}")

## E3 — Token Bucket Rate Limiting (FinOps & Quota Enforcement)

### 📚 Textbook Definition
The **Token Bucket** algorithm models capacity as a bucket that continuously refills tokens at a fixed rate `r` tokens/sec up to a burst capacity `B`. Requests take tokens and are rejected (`429`) if insufficient tokens remain.

### 🧠 Intuition
GenAI services have two quotas: Requests Per Minute (RPM) and Tokens Per Minute (TPM). Token bucket handles burstiness gracefully while enforcing strict throughput limits.

In [ ]:
import time

class TokenBucket:
    def __init__(self, capacity: float, refill_rate_per_sec: float):
        self.capacity = float(capacity)
        self.refill_rate = float(refill_rate_per_sec)
        self.tokens = float(capacity)
        self.last_update = time.monotonic()

    def consume(self, tokens: float = 1.0) -> bool:
        now = time.monotonic()
        elapsed = now - self.last_update
        self.last_update = now
        
        # Add newly generated tokens up to capacity
        self.tokens = min(self.capacity, self.tokens + (elapsed * self.refill_rate))
        
        if self.tokens >= tokens:
            self.tokens -= tokens
            return True
        return False

# Test Token Bucket
bucket = TokenBucket(capacity=5, refill_rate_per_sec=2.0) # 5 burst tokens, 2 tokens/sec
print("Initial burst consumes:")
for i in range(7):
    allowed = bucket.consume(1.0)
    print(f"  Request {i+1}: Allowed = {allowed} (Remaining: {bucket.tokens:.2f})")

time.sleep(1.0) # Wait 1 sec -> should refill 2 tokens
print("After 1s refill:")
print("  Request 8: Allowed =", bucket.consume(1.0))
print("  Request 9: Allowed =", bucket.consume(1.0))
print("  Request 10: Allowed =", bucket.consume(1.0))

## E4 — Exponential Backoff with Jitter & Idempotency Keys

### 📚 Textbook Definition
When upstream LLMs return transient errors (429 or 503), retrying with exponential backoff $T = 2^i$ and full jitter $rand(0, T)$ prevents the "thundering herd" problem. Idempotency keys ensure repeated calls do not charge tokens or duplicate side effects.

In [ ]:
import random
import asyncio

class IdempotentGateway:
    def __init__(self):
        self._cache = {} # idempotency_key -> response

    async def execute(self, idempotency_key: str, prompt: str) -> dict:
        if idempotency_key in self._cache:
            return {"data": self._cache[idempotency_key], "status": "HIT_IDEMPOTENT"}

        # Retry loop with exponential backoff & full jitter
        max_attempts = 4
        base_delay = 0.05
        
        for attempt in range(max_attempts):
            try:
                # Simulate intermittent 429 on first attempt
                if attempt == 0:
                    raise ConnectionResetError("429 Rate Limit Exceeded")
                
                # Successful call
                result = f"LLM Output for '{prompt}'"
                self._cache[idempotency_key] = result
                return {"data": result, "status": "SUCCESS", "attempts": attempt + 1}
                
            except Exception as e:
                if attempt == max_attempts - 1:
                    raise
                # Exponential backoff + full jitter
                backoff = base_delay * (2 ** attempt)
                sleep_time = random.uniform(0, backoff)
                await asyncio.sleep(sleep_time)

gw = IdempotentGateway()
first_run = await gw.execute("key-req-001", "Summarize contract")
print("First call :", first_run)
second_run = await gw.execute("key-req-001", "Summarize contract")
print("Second call:", second_run)

## E5 — High-Performance In-Memory Caching & LRU Eviction

### 📚 Textbook Definition
Caching LLM responses saves latency (sub-1ms vs 2000ms) and eliminates dollar cost. Least Recently Used (LRU) eviction ensures memory consumption is bounded under heavy traffic.

In [ ]:
from collections import OrderedDict
import hashlib

class BoundedLRUCache:
    def __init__(self, max_items: int = 100):
        self.max_items = max_items
        self._cache = OrderedDict()

    def _hash_key(self, prompt: str, model: str) -> str:
        raw = f"{model}:{prompt.strip().lower()}"
        return hashlib.sha256(raw.encode()).hexdigest()

    def get(self, prompt: str, model: str) -> Optional[str]:
        key = self._hash_key(prompt, model)
        if key not in self._cache:
            return None
        self._cache.move_to_end(key) # Mark as recently used
        return self._cache[key]

    def put(self, prompt: str, model: str, response: str):
        key = self._hash_key(prompt, model)
        if key in self._cache:
            self._cache.move_to_end(key)
        self._cache[key] = response
        if len(self._cache) > self.max_items:
            evicted = self._cache.popitem(last=False) # Evict oldest

cache = BoundedLRUCache(max_items=2)
cache.put("What is python?", "gpt-4o", "A high level programming language")
cache.put("What is RAG?", "gpt-4o", "Retrieval-Augmented Generation")
print("Get 'What is python?':", cache.get("What is python?", "gpt-4o") is not None)

# Add 3rd item -> triggers eviction of least recently used
cache.put("What is an agent?", "gpt-4o", "An autonomous reasoner")
print("After 3rd insert:")
print("  'What is RAG?' cached :", cache.get("What is RAG?", "gpt-4o") is not None)
print("  'What is python?' cached :", cache.get("What is python?", "gpt-4o") is not None)

## E6 — Background Workers & Producer-Consumer Queues

### 📚 Textbook Definition
Asynchronous task queues decouple synchronous user request ingestion from slow batch background jobs (e.g. document ingestion, vector index rebuilds, fine-tuning jobs).

In [ ]:
import asyncio

async def background_worker(worker_id: int, queue: asyncio.Queue):
    while True:
        job = await queue.get()
        if job is None: # Poison pill to shutdown
            queue.task_done()
            break
        doc_id, text = job
        # Simulate processing time
        await asyncio.sleep(0.02)
        print(f"  [Worker {worker_id}] Ingested Doc {doc_id} ({len(text)} chars)")
        queue.task_done()

# Producer pipeline
job_queue = asyncio.Queue(maxsize=10)
workers = [asyncio.create_task(background_worker(i, job_queue)) for i in range(2)]

# Enqueue 5 document ingestion jobs
for i in range(5):
    await job_queue.put((i + 1, f"Contract document contents for lease agreement #{i+1}"))

# Wait for all jobs to complete
await job_queue.join()

# Stop workers gracefully
for _ in workers:
    await job_queue.put(None)
await asyncio.gather(*workers)
print("All background worker ingestion jobs completed successfully!")

## 🎯 Section E — MAANG Interview Prep

**Q: How do you design an API gateway that handles 50,000 requests/minute to LLM providers while keeping costs strictly controlled?**
> A strong engineer explains:
> 1. **Token Bucket Rate Limiting** at tenant level to prevent noisy neighbours.
> 2. **Exact & Semantic Caching** (exact SHA256 match first, then vector similarity threshold >0.96) to absorb 30–50% of duplicate traffic with 0 token cost.
> 3. **Connection Pooling & Asynchronous ASGI (FastAPI/uvicorn)** to hold thousands of concurrent waiting connections with minimal RAM.
> 4. **Adaptive Model Routing (FinOps)**: routing simple questions to $0.15/1M token models and complex reasoning to $5.00/1M token models.

# 🤖 SECTION F — GENERATIVE AI CORE ENGINEERING

Here we implement the full spectrum of production Generative AI capabilities from first principles in pure, clean Python: LLM client abstractions, token-by-token streaming, structured outputs, tool calling, embeddings, end-to-end RAG, autonomous agents, and model routing.

---

## F1 — Unified LLM Client Architecture

### 📚 Textbook Definition
A unified client decouples vendor-specific SDK quirks (OpenAI, Anthropic, Gemini, Ollama) behind a consistent interface. It handles parameter normalization (`temperature`, `max_tokens`), error wrapping, and latency logging.

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Dict, Any, AsyncIterator
import asyncio

class BaseLLMClient(ABC):
    @abstractmethod
    async def complete(self, messages: List[Dict[str, str]], **kwargs) -> str:
        """Return raw string response."""
        pass
        
    @abstractmethod
    async def stream(self, messages: List[Dict[str, str]], **kwargs) -> AsyncIterator[str]:
        """Yield token chunks as they arrive."""
        pass

class MockEnterpriseLLM(BaseLLMClient):
    def __init__(self, model_name: str = "gpt-4o"):
        self.model_name = model_name

    async def complete(self, messages: List[Dict[str, str]], **kwargs) -> str:
        await asyncio.sleep(0.02) # Simulate network RTT
        user_msg = messages[-1]["content"]
        return f"Synthesized analysis of: '{user_msg}' by {self.model_name}"

    async def stream(self, messages: List[Dict[str, str]], **kwargs) -> AsyncIterator[str]:
        full_text = f"Token-by-token streamed inference for query."
        for word in full_text.split():
            await asyncio.sleep(0.01) # Simulate token latency
            yield word + " "

client = MockEnterpriseLLM("gpt-4o")
full_res = await client.complete([{"role": "user", "content": "Explain quantum computing"}])
print("Complete Response:", full_res)

## F2 — Server-Sent Events (SSE) & Token-by-Token Streaming

### 📚 Textbook Definition
Streaming delivers response tokens over an open HTTP connection via chunked transfer encoding (`text/event-stream`). This reduces **Time To First Token (TTFT)** from seconds to milliseconds.

In [ ]:
import sys

print("Simulating real-time UI streaming output: ", end="")
async for token in client.stream([{"role": "user", "content": "Tell me a joke"}]):
    sys.stdout.write(token)
    sys.stdout.flush()
print("\n[Stream completed]")

## F3 — Structured Outputs & Guaranteed JSON Schemas

### 📚 Textbook Definition
Standard LLMs return freeform unstructured text. Enterprise workflows (extracting invoices, booking appointments, routing queries) require **guaranteed adherence** to a strict Pydantic schema. If validation fails, an automatic repair retry loop is triggered.

In [ ]:
from pydantic import BaseModel, Field
import json

class ContractRiskExtraction(BaseModel):
    liability_cap_usd: float = Field(description="Maximum liability in USD")
    governing_law: str = Field(description="State or country jurisdiction")
    risk_level: str = Field(pattern="^(LOW|MEDIUM|HIGH|CRITICAL)$")
    requires_legal_review: bool

def parse_with_repair_loop(raw_llm_json: str, target_cls: type[BaseModel]) -> BaseModel:
    try:
        data = json.loads(raw_llm_json)
        return target_cls.model_validate(data)
    except Exception as e:
        print(f"  [Validation Failed: {e}] -> Triggering JSON Repair Loop...")
        # Simulated repair pass (e.g. stripping markdown backticks or fixing types)
        cleaned = raw_llm_json.replace("```json", "").replace("```", "").strip()
        data = json.loads(cleaned)
        # Type coercions if needed
        if "liability_cap_usd" in data and isinstance(data["liability_cap_usd"], str):
            data["liability_cap_usd"] = float(data["liability_cap_usd"].replace("$", "").replace(",", ""))
        return target_cls.model_validate(data)

# Test with slightly malformed LLM response
malformed_llm_output = '''```json
{
    "liability_cap_usd": "1,000,000",
    "governing_law": "Delaware",
    "risk_level": "HIGH",
    "requires_legal_review": true
}
```'''

parsed = parse_with_repair_loop(malformed_llm_output, ContractRiskExtraction)
print("Successfully extracted & validated structured output:")
print("  Liability Cap :", parsed.liability_cap_usd)
print("  Risk Level    :", parsed.risk_level)
print("  Review Needed :", parsed.requires_legal_review)

## F4 — Tool Calling & Dynamic Function Dispatch

### 📚 Textbook Definition
Function calling enables LLMs to decide when to call external Python functions, formulate their arguments in JSON according to a JSON Schema, and incorporate execution results back into their context.

In [ ]:
import inspect

class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, func):
        self._tools[func.__name__] = func
        return func

    def get_tool_schema(self, func_name: str) -> dict:
        func = self._tools[func_name]
        sig = inspect.signature(func)
        params = {}
        for name, p in sig.parameters.items():
            params[name] = {"type": "string" if p.annotation == str else "number"}
        return {
            "name": func_name,
            "description": func.__doc__.strip(),
            "parameters": {"type": "object", "properties": params, "required": list(params.keys())}
        }

    def execute(self, tool_name: str, arguments: dict) -> Any:
        if tool_name not in self._tools:
            raise ValueError(f"Tool {tool_name} not found")
        return self._tools[tool_name](**arguments)

registry = ToolRegistry()

@registry.register
def get_exchange_rate(base_currency: str, target_currency: str) -> float:
    """Fetches the current real-time foreign exchange rate between two currencies."""
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "GBP"): 0.79}
    return rates.get((base_currency.upper(), target_currency.upper()), 1.0)

print("Generated Tool Schema for LLM:")
print(json.dumps(registry.get_tool_schema("get_exchange_rate"), indent=2))
print("\nDispatched Tool Execution Result:", registry.execute("get_exchange_rate", {"base_currency": "USD", "target_currency": "EUR"}))

## F5 — Vector Math, Embeddings & Cosine Similarity

### 📚 Textbook Definition
Embeddings map unstructured text into continuous vector spaces $\mathbb{R}^d$ where semantic similarity equals geometric proximity. Cosine similarity measures directional alignment:
$$\cos(\mathbf{u}, \mathbf{v}) = rac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

In [ ]:
import math
from typing import List

def cosine_similarity(u: List[float], v: List[float]) -> float:
    """Calculates cosine similarity between two dense vectors from first principles."""
    dot = sum(a * b for a, b in zip(u, v))
    norm_u = math.sqrt(sum(a * a for a in u))
    norm_v = math.sqrt(sum(b * b for b in v))
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return dot / (norm_u * norm_v)

# Mock semantic embeddings for three concepts
# Vector components simulate: [finance, legal, python, food]
v_contract = [0.85, 0.95, 0.05, 0.01]
v_lease    = [0.80, 0.90, 0.02, 0.00]
v_pizza    = [0.01, 0.00, 0.05, 0.99]

print(f"Similarity (Contract, Lease) : {cosine_similarity(v_contract, v_lease):.4f} (High semantic match)")
print(f"Similarity (Contract, Pizza) : {cosine_similarity(v_contract, v_pizza):.4f} (Orthogonal / Unrelated)")

## F6 — Complete Production RAG Pipeline from Scratch

### 📚 Textbook Definition
Retrieval-Augmented Generation (RAG) grounds LLM outputs in verified external knowledge:
1. **Chunking**: Break large documents into semantic blocks with sliding overlap.
2. **Embedding**: Convert chunks to dense vectors.
3. **Retrieval**: Perform Top-K nearest neighbor search.
4. **Augmentation**: Inject retrieved contexts into the prompt.
5. **Generation**: LLM produces grounded answer citing source IDs.

In [ ]:
class SimpleRAGPipeline:
    def __init__(self):
        self.chunks = []
        self.vectors = []

    def chunk_document(self, doc_text: str, chunk_size: int = 10, overlap: int = 3) -> List[str]:
        words = doc_text.split()
        chunks = []
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunks.append(" ".join(words[start:end]))
            if end == len(words):
                break
            start += (chunk_size - overlap)
        return chunks

    def mock_embed(self, text: str) -> List[float]:
        # Deterministic bag-of-words pseudo-embedding for simulation
        words = text.lower().split()
        dim_python = sum(1 for w in words if "python" in w or "gil" in w or "async" in w)
        dim_legal  = sum(1 for w in words if "contract" in w or "liability" in w or "law" in w)
        dim_cloud  = sum(1 for w in words if "fastapi" in w or "aws" in w or "docker" in w)
        mag = math.sqrt(dim_python**2 + dim_legal**2 + dim_cloud**2) + 0.001
        return [dim_python/mag, dim_legal/mag, dim_cloud/mag]

    def index(self, documents: List[str]):
        for doc in documents:
            chunks = self.chunk_document(doc)
            for c in chunks:
                self.chunks.append(c)
                self.vectors.append(self.mock_embed(c))

    def retrieve(self, query: str, top_k: int = 2) -> List[tuple[str, float]]:
        q_vec = self.mock_embed(query)
        scored = [(self.chunks[i], cosine_similarity(q_vec, self.vectors[i])) for i in range(len(self.chunks))]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def generate_grounded(self, query: str) -> str:
        hits = self.retrieve(query, top_k=2)
        context = "\n---\n".join(f"[Snippet]: {h[0]}" for h in hits)
        return f"Grounded Answer to '{query}':\nUsing Context:\n{context}\nResult: Fully synthesized response citing retrieved snippets."

rag = SimpleRAGPipeline()
docs = [
    "Python GIL prevents true multi-core threading for CPU tasks. Use multiprocessing or async.",
    "The contract liability cap is strictly limited to 5 million dollars under New York governing law.",
    "FastAPI delivers high-throughput asynchronous endpoints using Starlette and Pydantic validation."
]
rag.index(docs)
print(rag.generate_grounded("What is the contract liability limit?"))

## F7 — Autonomous ReAct Agent Loop with Guardrails

### 📚 Textbook Definition
A **ReAct** (Reasoning + Acting) Agent interleaves reasoning traces (**Thought**) with actions (**Tool Call**) and observations (**Tool Output**) until reaching a **Final Answer**. Loop limits prevent infinite loops and runaway billing.

In [ ]:
class ReActAgent:
    def __init__(self, tools: ToolRegistry, max_steps: int = 5):
        self.tools = tools
        self.max_steps = max_steps

    async def run(self, goal: str) -> str:
        history = [f"Goal: {goal}"]
        step = 0
        
        while step < self.max_steps:
            step += 1
            print(f"  [Step {step}] Thinking...")
            
            # Step 1: Agent reasons that it needs the exchange rate
            if step == 1:
                thought = "Thought: To convert 1000 USD to EUR, I need the exchange rate."
                action = ("get_exchange_rate", {"base_currency": "USD", "target_currency": "EUR"})
                print(f"  {thought}")
                print(f"  Action: Call {action[0]} with {action[1]}")
                
                # Execute tool
                obs = self.tools.execute(action[0], action[1])
                history.append(f"Observation: {obs}")
                print(f"  Observation: Exchange rate is {obs}")
                continue
                
            # Step 2: Agent synthesizes final answer
            if step == 2:
                rate = 0.92
                final_amount = 1000 * rate
                return f"Final Answer: 1,000 USD equals {final_amount:.2f} EUR (at rate {rate})."
                
        return "Failed: Maximum reasoning steps exceeded."

agent = ReActAgent(registry)
final_result = await agent.run("Convert 1000 USD to EUR")
print("\nAgent Execution Outcome:")
print(final_result)

## F8 — Dynamic Model Routing (FinOps & Cost Optimization)

### 📚 Textbook Definition
Routing queries by complexity saves up to 80% on inference bills:
- **Tier 1 (Small / Cheap)**: Low-complexity questions, classifications, intent checks ($0.15/1M tokens).
- **Tier 2 (Large / Frontier)**: Multi-step reasoning, mathematical logic, complex coding ($5.00/1M tokens).

In [ ]:
def route_query(prompt: str) -> str:
    """Determines most cost-effective model based on query heuristic / complexity classifier."""
    complex_triggers = ["compare", "prove", "step-by-step", "architect", "synthesize", "refactor", "algorithm"]
    is_complex = any(w in prompt.lower() for w in complex_triggers) or len(prompt.split()) > 50
    
    if is_complex:
        return "claude-3-5-sonnet (High-Reasoning Tier - $3.00/1M)"
    return "gpt-4o-mini (Cost-Optimized Tier - $0.15/1M)"

queries = [
    "What is the capital of France?",
    "Step-by-step mathematically prove that quicksort average time complexity is O(N log N)",
    "Translate 'hello' to Spanish",
    "Architect an enterprise multi-region active-active distributed vector database"
]

print("FinOps Dynamic Model Routing:")
for q in queries:
    print(f"  Query: '{q[:40]}...' -> Selected: {route_query(q)}")

## 🎯 Section F — MAANG Interview Prep

**Q: In production RAG, why is vector search alone insufficient, and how do you build a hybrid search pipeline?**
> A strong engineer explains:
> Vector search excels at conceptual/semantic matches but struggles with exact keyword queries (part numbers, error codes, specific names). A production architecture uses **Hybrid Search**:
> 1. Dense retrieval (vector similarity) + Sparse retrieval (BM25 keyword search).
> 2. **Reciprocal Rank Fusion (RRF)** or a **Cross-Encoder Re-ranker** (e.g. Cohere Re-rank) to re-score the merged Top-50 candidates into the final Top-5 context window.

# 🏗️ SECTION G — PRODUCTION, OBSERVABILITY & RELIABILITY

Operating GenAI in production requires enterprise-grade observability: structured JSON logging, latency distribution histograms (p50/p95/p99), circuit breakers, performance profiling, and distributed systems fundamentals.

---

## G1 — Structured JSON Logging & Distributed Trace Propagation

### 📚 Textbook Definition
Unstructured `print()` statements are unacceptable in production. Logs must be structured JSON lines containing timestamp, level, `trace_id`, `request_id`, user identity, model name, token usage, and latency.

In [ ]:
import json
import logging
import uuid
from datetime import datetime, timezone

class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        log_obj = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
            "logger": record.name,
            "trace_id": getattr(record, "trace_id", "none"),
            "latency_ms": getattr(record, "latency_ms", None),
            "tokens": getattr(record, "tokens", None)
        }
        return json.dumps({k: v for k, v in log_obj.items() if v is not None})

logger = logging.getLogger("genai.gateway")
logger.setLevel(logging.INFO)
# Clear existing handlers if any
logger.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger.addHandler(handler)

# Simulate structured log event
trace = str(uuid.uuid4())
logger.info("Inference request dispatched", extra={"trace_id": trace, "latency_ms": 142.5, "tokens": 420})

## G2 — Latency Distributions & SLA Enforcement (p50, p95, p99)

### 📚 Textbook Definition
Mean latency is misleading in GenAI because long tail generations skew distributions. SLAs must be tracked using percentiles:
- **p50 (Median)**: Typical user experience.
- **p95 / p99**: The slowest 5% and 1% of requests (multi-turn reasoning, large context windows, queue stalls).

In [ ]:
import numpy as np

def calculate_percentiles(latencies_ms: List[float]) -> dict:
    """Calculates p50, p90, p95, p99 metrics."""
    arr = np.array(latencies_ms)
    return {
        "count": len(arr),
        "mean_ms": round(float(np.mean(arr)), 2),
        "p50_ms": round(float(np.percentile(arr, 50)), 2),
        "p90_ms": round(float(np.percentile(arr, 90)), 2),
        "p95_ms": round(float(np.percentile(arr, 95)), 2),
        "p99_ms": round(float(np.percentile(arr, 99)), 2),
    }

# Simulate realistic right-skewed LLM latency distribution
np.random.seed(42)
simulated_latencies = list(np.random.lognormal(mean=5.5, sigma=0.6, size=500)) # ~250ms median with long tail
metrics = calculate_percentiles(simulated_latencies)
print("GenAI Inference SLA Percentiles:")
for k, v in metrics.items():
    print(f"  {k:10s} : {v}")

## G3 — The Circuit Breaker Pattern

### 📚 Textbook Definition
When an external LLM provider experiences an outage, continuing to bombard it exhausts server threads and hangs user requests. A **Circuit Breaker** tracks failure rates:
- **CLOSED**: Normal operation. Requests pass through.
- **OPEN**: Failures exceeded threshold. Fails immediately without hitting provider; routes to fallback model.
- **HALF-OPEN**: Probe call after cooldown period to test provider recovery.

In [ ]:
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, recovery_time_sec: float = 1.0):
        self.failure_threshold = failure_threshold
        self.recovery_time_sec = recovery_time_sec
        self.failure_count = 0
        self.state = "CLOSED" # CLOSED, OPEN, HALF-OPEN
        self.last_state_change = time.monotonic()

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.last_state_change = time.monotonic()

    def can_attempt(self) -> bool:
        if self.state == "CLOSED":
            return True
        if self.state == "OPEN":
            if time.monotonic() - self.last_state_change >= self.recovery_time_sec:
                self.state = "HALF-OPEN"
                return True
            return False
        if self.state == "HALF-OPEN":
            return True
        return False

cb = CircuitBreaker(failure_threshold=2, recovery_time_sec=0.2)
print("State 1 (Normal):", cb.state)
cb.record_failure()
cb.record_failure()
print("State 2 (After 2 failures):", cb.state, "| Can call?:", cb.can_attempt())
time.sleep(0.25)
print("State 3 (After recovery window):", cb.state, "| Can call?:", cb.can_attempt())
cb.record_success()
print("State 4 (After successful probe):", cb.state)

## G4 — Profiling & Hot-Spot Analysis (`cProfile`)

### 📚 Textbook Definition
`cProfile` is Python's built-in deterministic C-extension profiler. It tracks execution count and total time spent per function to identify bottlenecks (e.g. regex chunking vs vector distance calculations).

In [ ]:
import cProfile
import pstats
import io

def simulate_pipeline_run():
    # Simulates repeated vector calculations and string processing
    vec1 = [0.1] * 128
    vec2 = [0.2] * 128
    for _ in range(2000):
        _ = sum(a * b for a, b in zip(vec1, vec2))
        _ = "enterprise document chunk with some text".replace("chunk", "block").split()

pr = cProfile.Profile()
pr.enable()
simulate_pipeline_run()
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('tottime')
ps.print_stats(5) # Top 5 slowest functions
print("Top 5 Bottlenecks from cProfile:")
for line in s.getvalue().splitlines()[:8]:
    print(" ", line)

## G5 — Distributed Systems & Scalability for GenAI

### 📚 Textbook Definition
Modern GenAI systems are distributed:
- **Stateless API tier**: Horizontally scalable across Kubernetes pods behind an L7 Load Balancer.
- **Stateful conversation store**: Redis / DynamoDB for session histories with TTL.
- **CAP Theorem trade-off**: RAG vector databases prioritize **Availability** and **Partition Tolerance** (AP) over strict consistency, using eventual consistency for vector index syncing.

---
## 🏆 GRAND GRADUATION CAPSTONE: Complete Enterprise GenAI System

This capstone integrates **everything** learned across Core Python, Internals, Advanced, Concurrency, Backend, GenAI, and Production:
1. **Pydantic DTOs & Validation**
2. **Token Bucket Rate Limiter**
3. **Semantic In-Memory Cache**
4. **Circuit Breaker with Fallback**
5. **ReAct Autonomous Agent**
6. **Structured JSON Output**
7. **Production Latency & Trace Logging**

In [ ]:
import time
import uuid

class EnterpriseGenAISystem:
    def __init__(self):
        self.rate_limiter = TokenBucket(capacity=10, refill_rate_per_sec=5.0)
        self.cache = BoundedLRUCache(max_items=50)
        self.circuit_breaker = CircuitBreaker(failure_threshold=3, recovery_time_sec=2.0)
        self.tools = registry

    async def execute_request(self, user_id: str, prompt: str) -> dict:
        trace_id = str(uuid.uuid4())[:8]
        t0 = time.perf_counter()

        # 1. Rate Limiting Check
        if not self.rate_limiter.consume(1.0):
            return {"trace_id": trace_id, "status": 429, "error": "Rate limit exceeded"}

        # 2. Cache Check
        cached_val = self.cache.get(prompt, "gpt-4o")
        if cached_val:
            latency = (time.perf_counter() - t0) * 1000
            return {"trace_id": trace_id, "status": 200, "cached": True, "latency_ms": round(latency, 2), "output": cached_val}

        # 3. Circuit Breaker & Execution
        if not self.circuit_breaker.can_attempt():
            # Graceful degradation fallback
            return {"trace_id": trace_id, "status": 200, "fallback": True, "output": "Service temporarily degraded: Using cached summary."}

        try:
            # Simulate agent reasoning execution
            res = self.tools.execute("get_exchange_rate", {"base_currency": "USD", "target_currency": "EUR"})
            output = f"Current exchange rate USD->EUR is {res}"
            self.circuit_breaker.record_success()
            self.cache.put(prompt, "gpt-4o", output)
            latency = (time.perf_counter() - t0) * 1000
            return {"trace_id": trace_id, "status": 200, "cached": False, "latency_ms": round(latency, 2), "output": output}
        except Exception as e:
            self.circuit_breaker.record_failure()
            return {"trace_id": trace_id, "status": 500, "error": str(e)}

system = EnterpriseGenAISystem()

# Run 1: Cache Miss
r1 = await system.execute_request("user_1", "What is the USD to EUR rate?")
print("Run 1 (Cache Miss):", r1)

# Run 2: Cache Hit
r2 = await system.execute_request("user_1", "What is the USD to EUR rate?")
print("Run 2 (Cache Hit) :", r2)
print(f"Latency speedup: {r1['latency_ms']:.2f}ms -> {r2['latency_ms']:.2f}ms")

## 🎯 Section G — MAANG Interview Prep

**Q: In an enterprise GenAI platform handling 100M tokens/day, how do you handle cascading failures when your primary model provider has a 20-second latency spike?**
> A strong engineer responds with a defense-in-depth architecture:
> 1. **Circuit Breakers**: Cut off calls immediately when timeouts exceed 5% over a 30-second window.
> 2. **Fallback Model Routing**: Degrade gracefully to secondary providers (e.g., fallback from Claude 3.5 Sonnet to GPT-4o-mini or a fine-tuned self-hosted Llama 3 checkpoint).
> 3. **Aggressive Hedging**: For VIP tier, if a request takes >3 seconds, launch a parallel hedge request to a secondary provider and cancel whichever finishes second.
> 4. **Tenant Isolation (Bulkhead Pattern)**: Prevent a single runaway customer query batch from exhausting the shared connection pool.